# Fine-tune PaddleOCR tiếng Việt - RTX 5090 Linux

Dùng trực tiếp `latin_PP-OCRv5_mobile_rec`, `MAX_TEXT_LENGTH=80`, ablation width 640 vs 960, rồi train cấu hình tốt nhất 4 epoch. Paddle chạy trong venv riêng; notebook kernel không import Paddle để tránh conflict package/CUDA.

## Cài đặt

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "pyyaml", "rapidfuzz", "pandas", "nbformat"], check=True)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['/workspace/.venv/bin/python', '-m', 'pip', 'install', '-q', 'gdown', 'pyyaml', 'rapidfuzz', 'pandas', 'nbformat'], returncode=0)

In [2]:
from pathlib import Path
import hashlib
import json
import math
import os
import pickle
import random
import shutil
import subprocess
import sys
import time
import unicodedata
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import yaml
from rapidfuzz.distance import Levenshtein
from IPython.display import display

SEED = 2026
MAX_TEXT_LENGTH = 80
ABLATION_EPOCHS = 1
FINAL_EPOCHS = 4
BATCH_SIZE = 16
IGNORE_SPACE = True

DRIVE_ID = "1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM"
PADDLEOCR_REF = "v3.3.0"
LATIN_MODEL = "latin_PP-OCRv5_mobile_rec"

In [3]:
ROOT = Path(os.environ.get("PADDLEOCR_ROOT", Path.cwd()))
WORK = ROOT / "paddleocr_vi_latin_5090"
PADDLE_DIR = WORK / "PaddleOCR"
DATA_EXTRACT = WORK / "data"
CONFIG_DIR = WORK / "configs"
OUTPUT_DIR = WORK / "output"
RESULTS_DIR = WORK / "results"
WEIGHT_DIR = WORK / "weights"
LOG_DIR = WORK / "logs"
VENV_DIR = WORK / ".venv_paddle5090"

for p in [WORK, DATA_EXTRACT, CONFIG_DIR, OUTPUT_DIR, RESULTS_DIR, WEIGHT_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "pyyaml", "rapidfuzz", "pandas"], check=True)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)


def supported_python():
    candidates = [sys.executable, shutil.which("python3.12"), shutil.which("python3.11"), shutil.which("python3.10"), shutil.which("python3.9")]
    seen = set()
    for candidate in candidates:
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)
        r = subprocess.run([candidate, "-c", "import sys; print(sys.version_info[:2])"], capture_output=True, text=True)
        if r.returncode == 0:
            version = eval(r.stdout.strip())
            if (3, 9) <= version <= (3, 12):
                return candidate
    raise RuntimeError("Cần Python 3.9-3.12 để tạo môi trường Paddle")


HOST_PYTHON = supported_python()
RUN_PYTHON = VENV_DIR / "bin/python"
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV.pop("PYTHONPATH", None)
RUNTIME_ENV["PYTHONNOUSERSITE"] = "1"
RUNTIME_ENV["VIRTUAL_ENV"] = str(VENV_DIR)
RUNTIME_ENV["PATH"] = str(VENV_DIR / "bin") + os.pathsep + RUNTIME_ENV.get("PATH", "")


def setup_runtime(recreate=False):
    if recreate and VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)
    if not RUN_PYTHON.exists():
        try:
            subprocess.run([HOST_PYTHON, "-m", "venv", str(VENV_DIR)], check=True)
        except subprocess.CalledProcessError:
            subprocess.run([HOST_PYTHON, "-m", "pip", "install", "-q", "virtualenv"], check=True)
            subprocess.run([HOST_PYTHON, "-m", "virtualenv", str(VENV_DIR)], check=True)

    subprocess.run([RUN_PYTHON, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True, env=RUNTIME_ENV)

    marker = VENV_DIR / ".paddle_ready"
    if not marker.exists():
        subprocess.run([
            RUN_PYTHON, "-m", "pip", "install", "-q",
            "paddlepaddle-gpu==3.2.1",
            "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu129/",
        ], check=True, env=RUNTIME_ENV)
        marker.touch()

    test = subprocess.run(
        [RUN_PYTHON, "-c", "import paddle; print(paddle.__version__)"],
        capture_output=True, text=True, env=RUNTIME_ENV,
    )
    if test.returncode != 0:
        print(test.stdout)
        print(test.stderr)
        if not recreate:
            return setup_runtime(recreate=True)
        raise RuntimeError("Không import được paddle trong venv sạch")

    check = subprocess.run(
        [RUN_PYTHON, "-c", "import paddle; paddle.utils.run_check()"],
        capture_output=True, text=True, env=RUNTIME_ENV,
    )
    print(check.stdout)
    if check.returncode != 0:
        print(check.stderr)
        raise RuntimeError("Paddle import được nhưng GPU run_check thất bại")
    return test.stdout.strip().splitlines()[-1]


PADDLE_VERSION = setup_runtime()
print("Runtime Python:", RUN_PYTHON)
print("Paddle:", PADDLE_VERSION)

if not PADDLE_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", PADDLEOCR_REF,
        "https://github.com/PaddlePaddle/PaddleOCR.git", str(PADDLE_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(PADDLE_DIR), "fetch", "--tags", "--force"], check=True)
    subprocess.run(["git", "-C", str(PADDLE_DIR), "checkout", PADDLEOCR_REF], check=True)

req_marker = VENV_DIR / ".paddleocr_requirements_ready"
if not req_marker.exists():
    subprocess.run([RUN_PYTHON, "-m", "pip", "install", "-q", "-r", str(PADDLE_DIR / "requirements.txt")], check=True, env=RUNTIME_ENV)
    req_marker.touch()

print("PaddleOCR:", subprocess.check_output(["git", "-C", str(PADDLE_DIR), "describe", "--tags", "--always"], text=True).strip())
print("Seed:", SEED)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Thu Aug 20 02:57:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:01:00.0 Off |                  N/A |
|  0%   34C    P8             25W /  600W |      18MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Cloning into '/workspace/SinoNom-NLP/paddleocr_vi_latin_5090/PaddleOCR'...
Note: switching to 'c2b0390d9a89ee76148be5c9d758a028527214d4'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false



PaddleOCR: v3.3.0
Seed: 2026


## Dữ liệu

In [4]:
import gdown

ZIP_PATH = WORK / "vi_rec_100k.zip"
if not ZIP_PATH.exists():
    gdown.download(id=DRIVE_ID, output=str(ZIP_PATH), quiet=False)


def marker_for(zip_path):
    zip_path = Path(zip_path)
    token = hashlib.sha1(str(zip_path.resolve()).encode("utf-8")).hexdigest()[:12]
    return zip_path.parent / f".{zip_path.stem}.{token}.extracted"


def unzip_once(zip_path, dest):
    zip_path = Path(zip_path)
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    marker = marker_for(zip_path)
    if marker.exists():
        return False
    print("Giải nén:", zip_path)
    with zipfile.ZipFile(zip_path, "r", allowZip64=True) as zf:
        zf.extractall(dest)
    marker.touch()
    return True


def valid_zip(path):
    return path.is_file() and "__MACOSX" not in path.parts and not path.name.startswith("._")


unzip_once(ZIP_PATH, DATA_EXTRACT)

while True:
    nested = [z for z in DATA_EXTRACT.rglob("*.zip") if valid_zip(z)]
    pending = [z for z in nested if not marker_for(z).exists()]
    if not pending:
        break
    print("Nested zip:", len(pending))
    for nested_zip in pending:
        unzip_once(nested_zip, nested_zip.parent)


def find_dataset_root(root):
    candidates = []
    for rec_train in Path(root).rglob("rec_train.txt"):
        parent = rec_train.parent
        if "__MACOSX" in parent.parts:
            continue
        required = [
            parent / "rec_train.txt",
            parent / "rec_val.txt",
            parent / "rec_test.txt",
            parent / "vi_dict.txt",
        ]
        if all(p.exists() for p in required):
            candidates.append(parent)

    if not candidates:
        txts = sorted(str(p) for p in Path(root).rglob("*.txt"))[:50]
        zips = sorted(str(p) for p in Path(root).rglob("*.zip"))[:50]
        raise FileNotFoundError(f"Không tìm thấy dataset root. TXT={txts} ZIP={zips}")

    candidates.sort(
        key=lambda p: (
            not all((p / split).is_dir() for split in ["train", "val", "test"]),
            len(p.parts),
        )
    )
    return candidates[0]


DATA_DIR = find_dataset_root(DATA_EXTRACT)
TRAIN_FILE = DATA_DIR / "rec_train.txt"
VAL_FILE = DATA_DIR / "rec_val.txt"
TEST_FILE = DATA_DIR / "rec_test.txt"
DICT_FILE = DATA_DIR / "vi_dict.txt"


def resolve_image_path(rel):
    rel = os.path.normpath(rel)
    direct = DATA_DIR / rel
    if direct.exists():
        return direct
    name = Path(rel).name
    matches = [DATA_DIR / split / name for split in ["train", "val", "test"] if (DATA_DIR / split / name).exists()]
    return matches[0] if len(matches) == 1 else direct


sample_line = next(x for x in TRAIN_FILE.read_text(encoding="utf-8").splitlines() if x.strip())
sample_rel = sample_line.split("\t", 1)[0]
sample_img = resolve_image_path(sample_rel)

print("Data root:", DATA_DIR)
print("Train dir:", DATA_DIR / "train")
print("Val dir:", DATA_DIR / "val")
print("Test dir:", DATA_DIR / "test")
print("Sample:", sample_rel, "->", sample_img, sample_img.exists())

Downloading...
From (original): https://drive.google.com/uc?id=1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM
From (redirected): https://drive.google.com/uc?id=1_NKW1CL49NKtnT92ddaNZwGcaFJOkUgM&confirm=t&uuid=704498f7-9c89-43b7-9217-dca0648e3ddf
To: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/vi_rec_100k.zip
100%|██████████| 3.81G/3.81G [01:35<00:00, 40.1MB/s]


Giải nén: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/vi_rec_100k.zip
Nested zip: 1
Giải nén: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/data/DACK/data/vi_rec_100k.zip
Data root: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/data/DACK/data
Train dir: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/data/DACK/data/train
Val dir: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/data/DACK/data/val
Test dir: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/data/DACK/data/test
Sample: train/letrieulichkhoatiensitap2__page_037_crop_14.jpg -> /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/data/DACK/data/train/letrieulichkhoatiensitap2__page_037_crop_14.jpg True


## Kiểm tra dữ liệu

In [5]:
def canonical_key(rel):
    p = Path(rel)
    if p.is_absolute():
        return os.path.normpath(os.path.relpath(p, DATA_DIR))
    return os.path.normpath(rel)


def read_labels(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n\r")
            if not line:
                continue
            rel, text = line.split("\t", 1)
            rows.append((canonical_key(rel), unicodedata.normalize("NFC", text)))
    return rows


train_rows = read_labels(TRAIN_FILE)
val_rows = read_labels(VAL_FILE)
test_rows = read_labels(TEST_FILE)
lengths = np.array([len(t) for _, t in train_rows])
actual_max = int(lengths.max())

stats = pd.DataFrame({
    "split": ["train", "val", "test"],
    "so_mau": [len(train_rows), len(val_rows), len(test_rows)],
})
display(stats)
print("Độ dài p50/p90/p95/p99/max:", np.percentile(lengths, [50, 90, 95, 99]).tolist(), actual_max)
print("MAX_TEXT_LENGTH cố định:", MAX_TEXT_LENGTH)

if actual_max > MAX_TEXT_LENGTH:
    raise ValueError(f"Label dài nhất {actual_max} > MAX_TEXT_LENGTH={MAX_TEXT_LENGTH}")

with open(DICT_FILE, "r", encoding="utf-8") as f:
    dict_chars = {unicodedata.normalize("NFC", x.rstrip("\n\r")) for x in f if x.rstrip("\n\r")}

all_chars = set("".join(t for _, t in train_rows + val_rows + test_rows)) - {" "}
missing_chars = sorted(all_chars - dict_chars)
missing_images = [rel for rel, _ in train_rows + val_rows + test_rows if not resolve_image_path(rel).exists()]

print("Ký tự ngoài vi_dict:", missing_chars[:50], "count=", len(missing_chars))
print("Ảnh không tìm thấy:", len(missing_images))
if missing_chars:
    raise ValueError("vi_dict.txt không phủ đủ ký tự")
if missing_images:
    raise FileNotFoundError(missing_images[:10])

,split,so_mau
0,train,93997
1,val,3000
2,test,3003


Độ dài p50/p90/p95/p99/max: [57.0, 68.0, 73.0, 79.0] 80
MAX_TEXT_LENGTH cố định: 80
Ký tự ngoài vi_dict: [] count= 0
Ảnh không tìm thấy: 0


## Model

In [6]:
MODEL_BASE = "https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model"
LATIN_CONFIG = PADDLE_DIR / "configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml"
LATIN_WEIGHT = WEIGHT_DIR / f"{LATIN_MODEL}_pretrained.pdparams"

if not LATIN_WEIGHT.exists():
    urllib.request.urlretrieve(f"{MODEL_BASE}/{LATIN_MODEL}_pretrained.pdparams", LATIN_WEIGHT)

print("Config:", LATIN_CONFIG)
print("Weight:", LATIN_WEIGHT)


def run_process(args, log_name):
    log_path = LOG_DIR / log_name
    start = time.time()
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.run(
            [str(x) for x in args],
            cwd=PADDLE_DIR,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            env=RUNTIME_ENV,
        )
    if p.returncode != 0:
        tail = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()[-80:]
        print("\n".join(tail))
        raise RuntimeError("Command failed: " + " ".join(map(str, args)))
    return time.time() - start


def run_infer(config_path, label_file, result_txt, log_name, pretrained=None, checkpoint=None):
    args = [
        RUN_PYTHON, "tools/infer_rec.py", "-c", config_path, "-o",
        f"Global.infer_img={DATA_DIR}",
        f"Global.infer_list={label_file}",
        f"Global.save_res_path={result_txt}",
        "Global.distributed=False",
    ]
    if pretrained is not None:
        args.append(f"Global.pretrained_model={pretrained}")
    if checkpoint is not None:
        args.append(f"Global.checkpoints={checkpoint}")
    return run_process(args, log_name)


def read_prediction_file(path):
    preds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            full_path, pred = parts[0], parts[1]
            rel = os.path.normpath(os.path.relpath(full_path, DATA_DIR))
            preds[rel] = unicodedata.normalize("NFC", pred)
    return preds


def metric_text(text):
    text = unicodedata.normalize("NFC", text)
    return text.replace(" ", "") if IGNORE_SPACE else text


def evaluate_predictions(label_file, pred_txt, jsonl_path=None):
    gt_rows = read_labels(label_file)
    preds = read_prediction_file(pred_txt)
    correct = 0
    distances = []
    records = []
    for rel, gt in gt_rows:
        pred = preds.get(rel, "")
        a, b = metric_text(pred), metric_text(gt)
        correct += int(a == b)
        distances.append(Levenshtein.normalized_distance(a, b))
        records.append({"image": rel, "gt": gt, "pred": pred})
    result = {
        "acc": correct / len(gt_rows),
        "norm_edit_dis": 1 - float(np.mean(distances)),
        "n": len(gt_rows),
        "missing_pred": sum(1 for rel, _ in gt_rows if rel not in preds),
    }
    if jsonl_path is not None:
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for r in records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return result, records


def make_baseline_config():
    with open(LATIN_CONFIG, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    cfg["Global"]["distributed"] = False
    cfg["Global"]["pretrained_model"] = str(LATIN_WEIGHT)
    cfg["Global"]["checkpoints"] = None
    for op in cfg["Eval"]["dataset"]["transforms"]:
        key = next(iter(op))
        if key == "RecResizeImg":
            op[key]["image_shape"] = [3, 48, 320]
    path = CONFIG_DIR / "latin_baseline.yml"
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return path

Config: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/PaddleOCR/configs/rec/PP-OCRv5/multi_language/latin_PP-OCRv5_mobile_rec.yml
Weight: /workspace/SinoNom-NLP/paddleocr_vi_latin_5090/weights/latin_PP-OCRv5_mobile_rec_pretrained.pdparams


## Baseline

In [7]:
BASELINE_CONFIG = make_baseline_config()
BASELINE_TXT = RESULTS_DIR / "baseline_test.txt"
BASELINE_JSONL = RESULTS_DIR / "pred_test_baseline.jsonl"

baseline_time = run_infer(
    BASELINE_CONFIG,
    TEST_FILE,
    BASELINE_TXT,
    "baseline_test.log",
    pretrained=LATIN_WEIGHT,
)
baseline_metric, baseline_records = evaluate_predictions(TEST_FILE, BASELINE_TXT, BASELINE_JSONL)
baseline_metric["inference_seconds"] = baseline_time
print("Baseline:", LATIN_MODEL)
display(pd.DataFrame([baseline_metric]))

Baseline: latin_PP-OCRv5_mobile_rec


,acc,norm_edit_dis,n,missing_pred,inference_seconds
0,0.047619,0.824196,3003,0,27.669346


## Fine-tune

In [8]:
CONFIGS = [
    {"name": "width_640", "width": 640},
    {"name": "width_960", "width": 960},
]


def make_config(spec, epochs, suffix="ablation"):
    with open(LATIN_CONFIG, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    name = f"{spec['name']}_{suffix}"
    width = int(spec["width"])
    save_dir = OUTPUT_DIR / name

    g = cfg["Global"]
    g["model_name"] = name
    g["epoch_num"] = int(epochs)
    g["seed"] = SEED
    g["distributed"] = False
    g["save_model_dir"] = str(save_dir)
    g["save_epoch_step"] = 1
    g["eval_batch_step"] = [0, 1000000]
    g["pretrained_model"] = str(LATIN_WEIGHT)
    g["checkpoints"] = None
    g["character_dict_path"] = str(DICT_FILE)
    g["max_text_length"] = MAX_TEXT_LENGTH
    g["use_space_char"] = True
    g["d2s_train_image_shape"] = [3, 48, width]

    cfg["Optimizer"]["lr"]["learning_rate"] = 0.0005
    cfg["Optimizer"]["lr"]["warmup_epoch"] = 0 if epochs <= 1 else 1
    cfg["Metric"]["ignore_space"] = IGNORE_SPACE

    for head in cfg["Architecture"]["Head"]["head_list"]:
        if "NRTRHead" in head:
            head["NRTRHead"]["max_text_length"] = MAX_TEXT_LENGTH

    train_ds = cfg["Train"]["dataset"]
    train_ds["data_dir"] = str(DATA_DIR)
    train_ds["label_file_list"] = [str(TRAIN_FILE)]
    for op in train_ds["transforms"]:
        key = next(iter(op))
        if key == "RecConAug":
            op[key]["image_shape"] = [48, width, 3]
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
        if key == "MultiLabelEncode":
            if op[key] is None:
                op[key] = {}
            op[key]["max_text_length"] = MAX_TEXT_LENGTH

    sampler = cfg["Train"]["sampler"]
    sampler["scales"] = [[width, 32], [width, 48], [width, 64]]
    sampler["first_bs"] = BATCH_SIZE
    sampler["fix_bs"] = True
    cfg["Train"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Train"]["loader"]["num_workers"] = 4

    eval_ds = cfg["Eval"]["dataset"]
    eval_ds["data_dir"] = str(DATA_DIR)
    eval_ds["label_file_list"] = [str(VAL_FILE)]
    for op in eval_ds["transforms"]:
        key = next(iter(op))
        if key == "RecResizeImg":
            op[key]["image_shape"] = [3, 48, width]
        if key == "MultiLabelEncode":
            if op[key] is None:
                op[key] = {}
            op[key]["max_text_length"] = MAX_TEXT_LENGTH
    cfg["Eval"]["loader"]["batch_size_per_card"] = BATCH_SIZE
    cfg["Eval"]["loader"]["num_workers"] = 4

    path = CONFIG_DIR / f"{name}.yml"
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
    return path, save_dir


ablation_config_paths = {}
for spec in CONFIGS:
    ablation_config_paths[spec["name"]] = make_config(spec, ABLATION_EPOCHS, "ablation")

display(pd.DataFrame(CONFIGS))

,name,width
0,width_640,640
1,width_960,960


In [9]:
def checkpoint_epoch(prefix):
    states = Path(str(prefix) + ".states")
    if not states.exists():
        return 0
    with open(states, "rb") as f:
        data = pickle.load(f)
    return int(data.get("epoch", 0))


def train_with_resume(cfg_path, save_dir, epochs, log_name):
    latest = save_dir / "latest"
    time_file = save_dir / "train_seconds.txt"
    total_seconds = float(time_file.read_text().strip()) if time_file.exists() else 0.0
    done_epoch = checkpoint_epoch(latest)
    if done_epoch < epochs:
        args = [RUN_PYTHON, "tools/train.py", "-c", cfg_path]
        if Path(str(latest) + ".pdparams").exists():
            args += ["-o", f"Global.checkpoints={latest}"]
        total_seconds += run_process(args, log_name)
        time_file.parent.mkdir(parents=True, exist_ok=True)
        time_file.write_text(str(total_seconds))
    return latest, total_seconds


ablation_results = []
for spec in CONFIGS:
    name = spec["name"]
    cfg_path, save_dir = ablation_config_paths[name]
    print("Training:", name)
    latest, train_seconds = train_with_resume(
        cfg_path, save_dir, ABLATION_EPOCHS, f"{name}_train.log"
    )
    pred_txt = RESULTS_DIR / f"{name}_val.txt"
    infer_seconds = run_infer(
        cfg_path, VAL_FILE, pred_txt, f"{name}_val.log", checkpoint=latest
    )
    metric, _ = evaluate_predictions(VAL_FILE, pred_txt)
    ablation_results.append({
        **spec,
        **metric,
        "train_seconds": train_seconds,
        "val_inference_seconds": infer_seconds,
        "config_path": str(cfg_path),
        "checkpoint": str(latest),
    })

ablation_df = pd.DataFrame(ablation_results).sort_values(
    ["acc", "norm_edit_dis"], ascending=[False, False]
).reset_index(drop=True)
ablation_df.to_csv(RESULTS_DIR / "ablation_width.csv", index=False)
display(ablation_df)

Training: width_640
Training: width_960


,name,width,acc,norm_edit_dis,n,missing_pred,train_seconds,val_inference_seconds,config_path,checkpoint
0,width_640,640,0.312667,0.936678,3000,0,502.341247,24.937643,/workspace/SinoNom-NLP/paddleocr_vi_latin_5090...,/workspace/SinoNom-NLP/paddleocr_vi_latin_5090...
1,width_960,960,0.085000,0.841633,3000,0,696.704613,24.890975,/workspace/SinoNom-NLP/paddleocr_vi_latin_5090...,/workspace/SinoNom-NLP/paddleocr_vi_latin_5090...


### So sánh 640 và 960

In [10]:
best_row = ablation_df.iloc[0]
best_spec = next(x for x in CONFIGS if x["name"] == best_row["name"])

with open(RESULTS_DIR / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_spec, f, ensure_ascii=False, indent=2)

print("Pretrained:", LATIN_MODEL)
print("MAX_TEXT_LENGTH:", MAX_TEXT_LENGTH)
print("Best width:", best_spec["width"])
print("Val acc:", best_row["acc"])
print("Val norm_edit_dis:", best_row["norm_edit_dis"])

Pretrained: latin_PP-OCRv5_mobile_rec
MAX_TEXT_LENGTH: 80
Best width: 640
Val acc: 0.31266666666666665
Val norm_edit_dis: 0.9366781298282782


## Fine-tune cuối

In [11]:
FINAL_CONFIG, FINAL_SAVE_DIR = make_config(best_spec, FINAL_EPOCHS, "final")
FINAL_CKPT, final_train_seconds = train_with_resume(
    FINAL_CONFIG, FINAL_SAVE_DIR, FINAL_EPOCHS, "best_final_train.log"
)

FINAL_VAL_TXT = RESULTS_DIR / "best_final_val.txt"
run_infer(FINAL_CONFIG, VAL_FILE, FINAL_VAL_TXT, "best_final_val.log", checkpoint=FINAL_CKPT)
final_val_metric, _ = evaluate_predictions(VAL_FILE, FINAL_VAL_TXT)
print(final_val_metric)

{'acc': 0.43166666666666664, 'norm_edit_dis': 0.9549063672128493, 'n': 3000, 'missing_pred': 0}


## Test

In [13]:
FINAL_TEST_TXT = RESULTS_DIR / "best_final_test.txt"
FINAL_TEST_JSONL = RESULTS_DIR / "pred_test_finetune.jsonl"

final_test_infer_seconds = run_infer(
    FINAL_CONFIG,
    TEST_FILE,
    FINAL_TEST_TXT,
    "best_final_test.log",
    checkpoint=FINAL_CKPT,
)
final_test_metric, final_test_records = evaluate_predictions(
    TEST_FILE,
    FINAL_TEST_TXT,
    FINAL_TEST_JSONL,
)

comparison = pd.DataFrame([
    {
        "model": f"Baseline: {LATIN_MODEL}",
        "acc": baseline_metric["acc"],
        "norm_edit_dis": baseline_metric["norm_edit_dis"],
        "train_seconds": 0.0,
    },
    {
        "model": "Fine-tune của nhóm",
        "acc": final_test_metric["acc"],
        "norm_edit_dis": final_test_metric["norm_edit_dis"],
        "train_seconds": final_train_seconds,
    },
])
comparison.to_csv(RESULTS_DIR / "comparison_test.csv", index=False)
display(comparison)

,model,acc,norm_edit_dis,train_seconds
0,Baseline: latin_PP-OCRv5_mobile_rec,0.047619,0.824196,0.000000
1,Fine-tune của nhóm,0.447885,0.956944,2017.803971


## Phân tích lỗi

In [14]:
TONE_MARKS = {"\u0300", "\u0301", "\u0303", "\u0309", "\u0323"}


def strip_tone(text):
    d = unicodedata.normalize("NFD", text)
    d = "".join(ch for ch in d if ch not in TONE_MARKS)
    return unicodedata.normalize("NFC", d)


def is_repeat_error(gt, pred):
    if len(pred) <= len(gt):
        return False
    for i in range(len(pred)):
        candidate = pred[:i] + pred[i + 1:]
        near_same = (i > 0 and pred[i] == pred[i - 1]) or (i + 1 < len(pred) and pred[i] == pred[i + 1])
        if candidate == gt and near_same:
            return True
    return False


def classify_error(gt, pred):
    if strip_tone(gt) == strip_tone(pred) and gt != pred:
        return "sai dấu thanh"
    if is_repeat_error(gt, pred):
        return "lặp ký tự"
    return "sai chữ cái"


wrong = [r for r in final_test_records if metric_text(r["gt"]) != metric_text(r["pred"])]
rng = random.Random(SEED)
sample20 = rng.sample(wrong, min(20, len(wrong)))

for r in sample20:
    r["loai_loi_goi_y"] = classify_error(r["gt"], r["pred"])

error_df = pd.DataFrame(sample20)
error_df.to_csv(RESULTS_DIR / "error_analysis_20.csv", index=False)
display(error_df)

error_pct = (
    error_df["loai_loi_goi_y"]
    .value_counts(normalize=True)
    .mul(100)
    .rename_axis("loai_loi")
    .reset_index(name="phan_tram")
)
display(error_pct)

,image,gt,pred,loai_loi_goi_y
0,test/lamsonthucluc__page_007_crop_24.jpg,Dâng sắc khắc bản in.,vâng sắc khắc bản in.,sai chữ cái
1,test/uctraitap__page_944_crop_7.jpg,trọi tục gọi gà gà ô. Hoa Triều thuộc về Đông ...,trọi tục gọi gà gà ồ. Hoa Triều thuộc về Đông ...,sai dấu thanh
2,test/MinhMenhchinhyeutap6ocr__page_147_crop_20...,Nam-thùy mọi việc đều đã tươm tất đầu ra đấy. ...,Nam thùy mọi việc đều đã tươm tất đâu ra đấy. ...,sai chữ cái
3,test/VNSuluocTranTrongKimv1_page_130_crop_2.jpg,Quang Cao đem 6 nghìn quân đóng sát thành. Quâ...,Quang Cao đem 6 nghìn quân áống sát thành. Quâ...,sai chữ cái
4,test/dinhtienhoang_page_015_crop_18.jpg,"giặc nó một trận kinh hồn thất đởm, nếu may mà...","giặc nó một trận kinh hồn thất đởm, nếu may mà...",sai chữ cái
5,test/sv_3896_crop_17.jpg,"Oai thàn khòng giét, ta cūng thě lòng tròi mó ...","Oai thần không giết, ta cũng thể lòng tròời mở...",sai chữ cái
6,test/daivietsukytucbien_page_442_crop_13.jpg,"Phúc Hợp làm Hữu phủ Kính quốc công, Khoa Thuy...","Phúc Hợp làm Hữru phủ Kính quốc công, Khoa Thu...",sai chữ cái
7,test/vanphamvietnamtrantrongkim_page_038_crop_...,Nào tôi cố nói thế bao giờ.,Nào tôi cò nói thế bao giờ.,sai chữ cái
8,test/sv_5165_crop_7.jpg,cái mới hiện đại với cái cũ lỗi thời trong ngh...,cái mới hiện đại với cái cũ lỗi thời trong ngh...,sai chữ cái
9,test/choichu_page_141_crop_19.jpg,Đất này thua trước đóng đô đây,Đất này thủa trước đóng đô đây,sai dấu thanh


,loai_loi,phan_tram
0,sai chữ cái,90.0
1,sai dấu thanh,10.0


## Kết quả

In [ ]:
summary = {
    "seed": SEED,
    "paddle_version": PADDLE_VERSION,
    "paddleocr_ref": PADDLEOCR_REF,
    "pretrained": LATIN_MODEL,
    "max_text_length": MAX_TEXT_LENGTH,
    "ablation_epochs": ABLATION_EPOCHS,
    "final_epochs": FINAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "best_config": best_spec,
    "baseline": baseline_metric,
    "final_val": final_val_metric,
    "final_test": final_test_metric,
    "final_train_seconds": final_train_seconds,
    "final_test_inference_seconds": final_test_infer_seconds,
}
with open(RESULTS_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

bundle = shutil.make_archive(str(WORK / "paddleocr_vi_results"), "zip", root_dir=RESULTS_DIR)
print("Results:", RESULTS_DIR)
print("Bundle:", bundle)
print("Final checkpoint:", FINAL_CKPT)